# Grilla sobre el Trial 2 (segundo mejor de la BO, que validó MEJOR que el Trial 1)

Hiperparámetros base: num_iterations=930, feature_fraction=0,154, num_leaves=1684, min_data_in_leaf=46, min_gain_to_split=0,216, lambda_l1=1,52, lambda_l2=7,55, bagging_fraction=0,843, bagging_freq=6. Cada combinación de la grilla busca su propio mejor corte (no uno fijo), ya que no validamos un corte único para este trial todavía.

### Seteo del ambiente — versión LOCAL

In [1]:
# rutas locales (variables protegidas, sobreviven a los rm(list=ls()) y a los PARAM <- list() de cada seccion)
PARAM_carpeta_datasets <- "C:/Users/Laura/Documents/DMEyF/datasets/"
PARAM_carpeta_exp      <- "C:/Users/Laura/Documents/DMEyF/exp/"
PARAM_archivo_crudo    <- "C:/Users/Laura/Documents/DMEyF/competencia_01_crudo (1).csv"

dir.create(PARAM_carpeta_datasets, showWarnings = FALSE, recursive = TRUE)
dir.create(PARAM_carpeta_exp, showWarnings = FALSE, recursive = TRUE)

file.exists(PARAM_archivo_crudo)  # tiene que dar TRUE

[1] TRUE

limpio el ambiente de R

In [2]:
# limpio la memoria
rm(list = setdiff(ls(all.names = TRUE), c("PARAM_carpeta_datasets", "PARAM_carpeta_exp", "PARAM_archivo_crudo"))) # conservo las rutas locales
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,656705,35.1,1447542,77.4,1419493,75.9
Vcells,1207206,9.3,8388608,64.0,1928501,14.8


In [3]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")

Caricamento del pacchetto richiesto: data.table

Caricamento del pacchetto richiesto: parallel

Caricamento del pacchetto richiesto: R.utils

Caricamento del pacchetto richiesto: R.oo

Caricamento del pacchetto richiesto: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Caricamento pacchetto: 'R.oo'


Il seguente oggetto è mascherato da 'package:R.methodsS3':

    throw


I seguenti oggetti sono mascherati da 'package:methods':

    getClasses, getMethods


I seguenti oggetti sono mascherati da 'package:base':

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Caricamento pacchetto: 'R.utils'


Il seguente oggetto è mascherato da 'package:utils':

    timestamp


I seguenti oggetti sono mascherati da 'package:base':

    cat, commandArgs, getOption, isOpen, nullfile, parse, use,

In [4]:
PARAM <- list()
PARAM$carpeta_datasets <- PARAM_carpeta_datasets
PARAM$carpeta_exp      <- PARAM_carpeta_exp
PARAM$experimento <- 4992
PARAM$semilla_primigenia <- 585581


In [5]:
# training y future
PARAM$train <- c(202104)
PARAM$train_final <- c(202104)
PARAM$future <- c(202106)
PARAM$semilla_kaggle <- 314159
PARAM$cortes <- seq(4000, 19000, by= 500)

In [6]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.1

In [7]:
# Parametros LightGBM
PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt",
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  min_sum_hessian_in_leaf= 0.001,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  num_iterations= 1200,
  learning_rate= 0.02,
  feature_fraction= 0.5,
  num_leaves= 750,
  min_data_in_leaf= 5000
)

In [8]:
# lectura del dataset
dataset <- fread(paste0(PARAM$carpeta_datasets, "competencia_01.csv"), stringsAsFactors= TRUE)

In [9]:
# los campos que se van a utilizar
# (uso "dataset" en vez de "dataset_train" -- ese objeto era de la fase de la BO, no lo recreamos aca)
campos_buenos <- setdiff(
  colnames(dataset),
  c("clase_ternaria", "clase01", "azar", "training")
)

# Salsa Magica (de Gustavo): saco las 2 variables con Data Drifting
campos_buenos <- setdiff(campos_buenos, c("cprestamos_personales", "mprestamos_personales"))

In [10]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [11]:
dataset_train <- dataset[foto_mes %in% PARAM$train_final]
dataset_train[,.N,clase_ternaria]

clase_ternaria,N
<fct>,<int>
CONTINUA,161181
BAJA+2,1139
BAJA+1,964


In [12]:
# dejo los datos en el formato que necesita LightGBM

dtrain_final <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

In [13]:
# iniciliazo el dataset de realidad, para medir ganancia
realidad_inicializar <- function( pfuture, pparam) {

  # datos para verificar la ganancia
  drealidad <- pfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]

  particionar(drealidad,
    division= c(3, 7),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_kaggle
  )

  return( drealidad )
}

In [14]:
# evaluo ganancia en los datos de la realidad

realidad_evaluar <- function( prealidad, pprediccion) {

  prealidad[ pprediccion,
    on= c("numero_de_cliente", "foto_mes"),
    predicted:= i.Predicted
  ]

  tbl <- prealidad[, list("qty"=.N), list(fold, predicted, clase_ternaria)]

  res <- list()
  res$public  <- tbl[fold==1 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.3
  res$private <- tbl[fold==2 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.7
  res$total <- tbl[predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]

  prealidad[, predicted:=NULL]
  return( res )
}

In [15]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [16]:
# aplico el modelo a los datos sin clase (el futuro real)
dfuture <- dataset[foto_mes %in% PARAM$future]

In [17]:
drealidad <- realidad_inicializar(dfuture, PARAM)

## Grilla: entreno con cada combinación de learning_rate y scale_pos_weight, aplico al futuro real

In [18]:
# hiperparametros del Trial 2 (segundo mejor de la BO), como base fija
PARAM$out$lgbm$mejores_hiperparametros <- list(
  num_iterations = 930,
  learning_rate = 0.02731257,   # se pisa en el loop
  feature_fraction = 0.1541961,
  num_leaves = 1684,
  min_data_in_leaf = 46,
  scale_pos_weight = 0.6223848,  # se pisa en el loop
  min_gain_to_split = 0.2158453,
  lambda_l1 = 1.520327,
  lambda_l2 = 7.548036,
  bagging_fraction = 0.8425105,
  bagging_freq = 6
)

param_final <- modifyList(PARAM$lgbm$param_fijos, PARAM$out$lgbm$mejores_hiperparametros)
param_final$metric <- "auc"
param_normalizado_base <- copy(param_final)
param_normalizado_base$min_data_in_leaf <- round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

# grilla: 5 valores de learning_rate x 5 de scale_pos_weight = 25 combinaciones
valores_lr  <- c(0.01, 0.02, 0.03, 0.05, 0.08)
valores_spw <- c(0.3, 0.5, 0.62, 1, 2)

cortes_grilla <- seq(4000, 19000, by= 500)

resultados_grilla_lr_spw <- data.table(learning_rate = numeric(), scale_pos_weight = numeric(), mejor_corte = integer(), TOTAL = numeric())

for (lr in valores_lr) {
  for (spw in valores_spw) {

    param_grilla <- copy(param_normalizado_base)
    param_grilla$learning_rate <- lr
    param_grilla$scale_pos_weight <- spw

    set.seed(PARAM$semilla_primigenia, "L'Ecuyer-CMRG")

    modelo_grilla <- lgb.train(
      data = dtrain_final,
      param = param_grilla
    )

    prediccion_grilla <- predict(modelo_grilla, data.matrix(dfuture[, campos_buenos, with = FALSE]))

    tb_pred_grilla <- dfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]
    tb_pred_grilla[, prob := prediccion_grilla]
    setorder(tb_pred_grilla, -prob)

    mejor_ganancia <- -Inf
    mejor_corte_local <- NA
    for (envios in cortes_grilla) {
      tb_pred_grilla[, Predicted := 0L]
      tb_pred_grilla[1:envios, Predicted := 1L]
      ganancia <- tb_pred_grilla[Predicted == 1L, sum(ifelse(clase_ternaria == "BAJA+2", 1072500, -27500))]
      if (ganancia > mejor_ganancia) {
        mejor_ganancia <- ganancia
        mejor_corte_local <- envios
      }
    }

    cat("learning_rate=", lr, " scale_pos_weight=", spw, " mejor_corte=", mejor_corte_local, " TOTAL=", mejor_ganancia, "\n", sep = "")

    resultados_grilla_lr_spw <- rbindlist(list(resultados_grilla_lr_spw,
      data.table(learning_rate = lr, scale_pos_weight = spw, mejor_corte = mejor_corte_local, TOTAL = mejor_ganancia)))
  }
}

fwrite(resultados_grilla_lr_spw, file = paste0(PARAM$carpeta_exp, "grilla_lr_spw_trial2.txt"), sep = "\t")
setorder(resultados_grilla_lr_spw, -TOTAL)
resultados_grilla_lr_spw

learning_rate=0.01 scale_pos_weight=0.3 mejor_corte=12500 TOTAL=442750000
learning_rate=0.01 scale_pos_weight=0.5 mejor_corte=12500 TOTAL=454850000
learning_rate=0.01 scale_pos_weight=0.62 mejor_corte=11500 TOTAL=451550000
learning_rate=0.01 scale_pos_weight=1 mejor_corte=12000 TOTAL=452100000
learning_rate=0.01 scale_pos_weight=2 mejor_corte=12000 TOTAL=448800000
learning_rate=0.02 scale_pos_weight=0.3 mejor_corte=10500 TOTAL=454850000
learning_rate=0.02 scale_pos_weight=0.5 mejor_corte=12000 TOTAL=456500000
learning_rate=0.02 scale_pos_weight=0.62 mejor_corte=10500 TOTAL=448250000
learning_rate=0.02 scale_pos_weight=1 mejor_corte=9500 TOTAL=439450000
learning_rate=0.02 scale_pos_weight=2 mejor_corte=12500 TOTAL=447150000
learning_rate=0.03 scale_pos_weight=0.3 mejor_corte=11500 TOTAL=451550000
learning_rate=0.03 scale_pos_weight=0.5 mejor_corte=10000 TOTAL=4.4e+08
learning_rate=0.03 scale_pos_weight=0.62 mejor_corte=11000 TOTAL=441100000
learning_rate=0.03 scale_pos_weight=1 mejor_co

learning_rate,scale_pos_weight,mejor_corte,TOTAL
<dbl>,<dbl>,<dbl>,<dbl>
0.02,0.50,12000,456500000
0.01,0.50,12500,454850000
0.02,0.30,10500,454850000
0.01,1.00,12000,452100000
0.01,0.62,11500,451550000
0.03,0.30,11500,451550000
0.01,2.00,12000,448800000
0.02,0.62,10500,448250000
0.02,2.00,12500,447150000


## Resumen de dispersión

In [19]:
cat("Ganador: learning_rate =", resultados_grilla_lr_spw[1, learning_rate],
    " scale_pos_weight =", resultados_grilla_lr_spw[1, scale_pos_weight],
    " con TOTAL =", resultados_grilla_lr_spw[1, TOTAL], "\n")

dcast(resultados_grilla_lr_spw, learning_rate ~ scale_pos_weight, value.var = "TOTAL")

Ganador: learning_rate = 0.02  scale_pos_weight = 0.5  con TOTAL = 456500000 


learning_rate,0.3,0.5,0.62,1,2
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0.01,442750000,454850000,451550000,452100000,448800000
0.02,454850000,456500000,448250000,439450000,447150000
0.03,451550000,440000000,441100000,433950000,438350000
0.05,441650000,429550000,425700000,419100000,425700000
0.08,440000000,423500000,413050000,409200000,408100000
